**Chatbot using Hugging Face Transformers.**

In [1]:
!pip install transformers torch

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [2]:
model_name = "microsoft/DialoGPT-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

print("Model Loaded Successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/641 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/351M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-small
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ Model Loaded Successfully


In [4]:
# Predefined knowledge base (clean + scalable)
knowledge_base = {
    "ai": "Artificial Intelligence is the simulation of human intelligence by machines.",
    "python": "Python was created by Guido van Rossum in 1991.",
    "hello": "Hello! Nice to meet you. How can I help you?"
}

In [6]:
# Function to Check Smart Replies
def get_smart_reply(user_input):
    user_input = user_input.lower()

    for key in knowledge_base:
        if key in user_input:
            return knowledge_base[key]

    return None

In [7]:
# Chatbot Function (Main Logic)

def chatbot():
    print("Chatbot: Hello! I am your AI assistant. Type 'exit' to stop.\n")

    chat_history_ids = None

    while True:
        user_input = input("You: ")

        # Exit condition
        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye! 👋")
            break

        # Check smart reply first
        smart_reply = get_smart_reply(user_input)

        if smart_reply:
            print("Chatbot:", smart_reply)
            continue

        # Transformer response
        new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1) if chat_history_ids is not None else new_input_ids

        chat_history_ids = model.generate(
            bot_input_ids,
            max_length=300,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            top_k=40,
            top_p=0.85,
            temperature=0.7
        )

        response = tokenizer.decode(
            chat_history_ids[:, bot_input_ids.shape[-1]:][0],
            skip_special_tokens=True
        )

        print("Chatbot:", response.strip())

In [8]:
chatbot()

Chatbot: Hello! I am your AI assistant. Type 'exit' to stop.

You: hi


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Chatbot: Hi
You: hello
Chatbot: Hello! Nice to meet you. How can I help you?
You: what is ml
Chatbot: It's like ML
You: What is AI?
Chatbot: Artificial Intelligence is the simulation of human intelligence by machines.
You: python
Chatbot: Python was created by Guido van Rossum in 1991.
You: joke
Chatbot: This is like ML
You: what.?
Chatbot: What's ML
You: exit
Chatbot: Goodbye! 👋
